In [1]:
!nvidia-smi
print("\n" + "="*70)
print("Installing dependencies...")
print("="*70)
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers>=4.38.0 peft>=0.9.0 bitsandbytes>=0.42.0 accelerate>=0.27.0
!pip install -q open-clip-torch>=2.24.0 datasets>=2.18.0 Pillow tqdm
print("\n[✓] All dependencies installed.")

Sun Nov 22 21:47:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 13.2     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:4E:00.0 Off |                   On |
| N/A   30C    P0             44W /  400W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import os, sys, gc, json, math, random, shutil, zipfile, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── CUDA checks ──────────────────────────────────────────────────────
assert torch.cuda.is_available(), "CUDA is not available! This notebook requires a GPU."
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_mem / (1024**3) if hasattr(torch.cuda.get_device_properties(0), 'total_mem') else torch.cuda.get_device_properties(0).total_memory / (1024**3)
BF16_OK = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16

print(f"GPU        : {GPU_NAME}")
print(f"VRAM       : {VRAM_GB:.1f} GB")
print(f"bfloat16   : {'Supported ✓' if BF16_OK else 'Not supported — using float16'}")
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA       : {torch.version.cuda}")
print(f"Seed       : {SEED}")
print("[✓] Environment verified.")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU        : NVIDIA A100-SXM4-40GB MIG 3g.20gb
VRAM       : 19.6 GB
bfloat16   : Supported ✓
PyTorch    : 2.14.1+cu130
CUDA       : 13.0
Seed       : 42
[✓] Environment verified.


In [3]:
# ══════════════════════════════════════════════════════════════════════
# MODEL IDENTIFIERS
# ══════════════════════════════════════════════════════════════════════
LLM_MODEL_ID    = "BioMistral/BioMistral-7B"
VISION_MODEL_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
DATASET_ID      = "eltorio/ROCO-radiology"

# ══════════════════════════════════════════════════════════════════════
# ARCHITECTURE DIMENSIONS
# ══════════════════════════════════════════════════════════════════════
VISION_DIM       = 512    # BiomedCLIP output dim
LLM_HIDDEN_DIM   = 4096   # BioMistral-7B hidden size
NUM_LAB_VALUES   = 20     # Continuous biomarker values
NUM_LAB_FLAGS    = 4      # Low, Normal, High, Critical
LAB_INPUT_DIM    = NUM_LAB_VALUES + NUM_LAB_FLAGS  # 24

# ══════════════════════════════════════════════════════════════════════
# QLoRA CONFIGURATION
# ══════════════════════════════════════════════════════════════════════
LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
LORA_TARGETS     = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ══════════════════════════════════════════════════════════════════════
# TRAINING HYPERPARAMETERS
# ══════════════════════════════════════════════════════════════════════
BATCH_SIZE       = 1
GRAD_ACCUM       = 16
MAX_SEQ_LEN      = 512    # Extended to 512 so prompt + findings + impression never truncate
GRAD_CLIP_NORM   = 1.0

# Stage 1 — Projector Warmup
STAGE1_EPOCHS    = 1
STAGE1_LR        = 5e-4

# Stage 2 — Joint QLoRA Fine-Tuning
STAGE2_EPOCHS    = 2      # 2 epochs over 30k samples (3,750 optimization steps)
STAGE2_LR        = 1.5e-4
STAGE2_MIN_LR    = 1e-5
WEIGHT_DECAY     = 0.01

# Dataset size (set to 30,000 samples)
MAX_SAMPLES      = 20000

# ══════════════════════════════════════════════════════════════════════
# CHECKPOINT PATHS
# ══════════════════════════════════════════════════════════════════════
CHECKPOINT_DIR = Path("checkpoints/best_radiology_gpt")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PROJECTORS_PATH  = CHECKPOINT_DIR / "projectors.pt"
LORA_DIR         = CHECKPOINT_DIR / "lora_adapter"
ZIP_PATH         = Path("best_radiology_gpt.zip")

print("[✓] Configuration loaded.")
print(f"    Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"    Max sequence length  : {MAX_SEQ_LEN}")
print(f"    Stage 1             : {STAGE1_EPOCHS} epoch(s) @ lr={STAGE1_LR}")
print(f"    Stage 2             : {STAGE2_EPOCHS} epoch(s) @ lr={STAGE2_LR} → {STAGE2_MIN_LR}")
print(f"    Max samples         : {MAX_SAMPLES:,} samples")

[✓] Configuration loaded.
    Effective batch size : 16
    Max sequence length  : 512
    Stage 1             : 1 epoch(s) @ lr=0.0005
    Stage 2             : 2 epoch(s) @ lr=0.00015 → 1e-05
    Max samples         : 20,000 samples


In [4]:
class VisionProjector(nn.Module):
    """Projects BiomedCLIP visual embeddings (512-dim) into the BioMistral
    token space (4096-dim) via a 2-layer MLP with GELU activation.

    Architecture:  Linear(512, 2048) → GELU → Dropout(0.1) → Linear(2048, 4096)
    """

    def __init__(self, vis_dim: int = 512, llm_dim: int = 4096, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vis_dim, 2048),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2048, llm_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, vis_dim) or (B, num_patches, vis_dim)
        out = self.net(x)
        if out.ndim == 2:
            out = out.unsqueeze(1)  # (B, 1, llm_dim)
        return out


class TabularLabProjector(nn.Module):
    """Projects 20 normalized lab biomarker values + 4 one-hot status flags
    (Low, Normal, High, Critical) into a single rich prefix token (4096-dim).

    Input vector dimension: 20 (continuous) + 4 (flags) = 24
    Architecture:  Linear(24, 1024) → GELU → Dropout(0.1) → Linear(1024, 4096)
    Output: (B, 1, 4096) — a single summary token per sample.
    """

    def __init__(self, input_dim: int = 24, llm_dim: int = 4096, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(1024, llm_dim),
        )

    def forward(self, lab_input: torch.Tensor) -> torch.Tensor:
        # lab_input: (B, 24) — 20 normalized values concatenated with 4 flags
        out = self.net(lab_input)  # (B, llm_dim)
        return out.unsqueeze(1)    # (B, 1, llm_dim)


# Quick shape verification
_v = VisionProjector(VISION_DIM, LLM_HIDDEN_DIM)
_t = TabularLabProjector(LAB_INPUT_DIM, LLM_HIDDEN_DIM)
assert _v(torch.randn(2, VISION_DIM)).shape == (2, 1, LLM_HIDDEN_DIM)
assert _t(torch.randn(2, LAB_INPUT_DIM)).shape == (2, 1, LLM_HIDDEN_DIM)
del _v, _t
print("[✓] VisionProjector  : Linear(512 → 2048) → GELU → Dropout → Linear(2048 → 4096)")
print("[✓] TabularLabProjector : Linear(24 → 1024) → GELU → Dropout → Linear(1024 → 4096)")

[✓] VisionProjector  : Linear(512 → 2048) → GELU → Dropout → Linear(2048 → 4096)
[✓] TabularLabProjector : Linear(24 → 1024) → GELU → Dropout → Linear(1024 → 4096)


In [5]:
import open_clip
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ──────────────────────────────────────────────────────────────────────
# 1. BiomedCLIP Vision Tower (Frozen)
# ──────────────────────────────────────────────────────────────────────
print("[*] Loading BiomedCLIP Vision Encoder...")
clip_model, _, preprocess_vis = open_clip.create_model_and_transforms(VISION_MODEL_ID)
vision_tower = clip_model.visual  # extract just the visual encoder
vision_tower = vision_tower.to(DEVICE).eval()
for param in vision_tower.parameters():
    param.requires_grad = False
del clip_model
gc.collect()
torch.cuda.empty_cache()
print(f"    Vision tower loaded and frozen. Output dim: {VISION_DIM}")

# ──────────────────────────────────────────────────────────────────────
# 2. BioMistral-7B in 4-bit NF4 Quantization
# ──────────────────────────────────────────────────────────────────────
print(f"[*] Loading {LLM_MODEL_ID} in 4-bit NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

base_llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
    trust_remote_code=True,
)
base_llm.config.use_cache = False  # Required for gradient checkpointing
base_llm = prepare_model_for_kbit_training(base_llm)
print(f"    LLM hidden_size: {base_llm.config.hidden_size}")

# ──────────────────────────────────────────────────────────────────────
# 3. Attach QLoRA Adapters
# ──────────────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
peft_llm = get_peft_model(base_llm, lora_config)
peft_llm.gradient_checkpointing_enable()
peft_llm.print_trainable_parameters()

# ──────────────────────────────────────────────────────────────────────
# 4. Instantiate Projectors on GPU
# ──────────────────────────────────────────────────────────────────────
vis_proj = VisionProjector(VISION_DIM, LLM_HIDDEN_DIM).to(device=DEVICE, dtype=COMPUTE_DTYPE)
lab_proj = TabularLabProjector(LAB_INPUT_DIM, LLM_HIDDEN_DIM).to(device=DEVICE, dtype=COMPUTE_DTYPE)

total_proj_params = sum(p.numel() for p in vis_proj.parameters()) + sum(p.numel() for p in lab_proj.parameters())
print(f"    Projector trainable params: {total_proj_params:,}")

vram_used = torch.cuda.memory_allocated() / (1024**3)
print(f"\n[✓] Full model loaded. Current VRAM usage: {vram_used:.2f} GB")

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-312-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-312-x86_64-linux-gnu.so
W1122 21:47:31.210000 25837 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


[*] Loading BiomedCLIP Vision Encoder...


    Vision tower loaded and frozen. Output dim: 512
[*] Loading BioMistral/BioMistral-7B in 4-bit NF4...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:06<00:00, 45.19it/s]


    LLM hidden_size: 4096
trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758
    Projector trainable params: 13,667,328

[✓] Full model loaded. Current VRAM usage: 4.83 GB


In [6]:
from datasets import load_dataset

# ══════════════════════════════════════════════════════════════════════
# Normal reference ranges for 20 clinical biomarkers
# (mean, std) — used for normalization and correlated synthesis
# ══════════════════════════════════════════════════════════════════════
LAB_REFERENCE = {
    # Hematology
    "WBC":       (7.0, 2.5),    # k/µL
    "RBC":       (4.8, 0.5),    # M/µL
    "HGB":       (14.5, 1.5),   # g/dL
    "HCT":       (42.0, 4.0),   # %
    "MCV":       (88.0, 6.0),   # fL
    "MCH":       (30.0, 2.0),   # pg
    "MCHC":      (34.0, 1.5),   # g/dL
    "PLT":       (250.0, 60.0), # k/µL
    "NEUT_PCT":  (60.0, 10.0),  # %
    "LYMPH_PCT": (30.0, 8.0),   # %
    # Chemistry
    "SODIUM":    (140.0, 3.0),  # mEq/L
    "POTASSIUM": (4.2, 0.4),    # mEq/L
    "CHLORIDE":  (102.0, 3.0),  # mEq/L
    "BICARB":    (24.0, 2.5),   # mEq/L
    "BUN":       (14.0, 4.0),   # mg/dL
    "CREATININE":(0.95, 0.2),   # mg/dL
    "GLUCOSE":   (95.0, 15.0),  # mg/dL
    "CALCIUM":   (9.5, 0.5),    # mg/dL
    # Inflammatory
    "CRP":       (1.0, 0.8),    # mg/L
    "ESR":       (10.0, 5.0),   # mm/hr
}
LAB_KEYS = list(LAB_REFERENCE.keys())

# ══════════════════════════════════════════════════════════════════════
# Clinical keyword → correlated lab perturbation profiles
# ══════════════════════════════════════════════════════════════════════
CLINICAL_PROFILES = {
    "bacterial_infection": {
        "keywords": ["pneumonia", "consolidation", "abscess", "empyema", "bronchograms", "lobar"],
        "perturbations": {
            "WBC": (2.0, 3.5), "NEUT_PCT": (1.2, 1.5), "LYMPH_PCT": (0.5, 0.8),
            "CRP": (20.0, 80.0), "ESR": (3.0, 6.0), "PLT": (1.2, 1.8),
        },
    },
    "viral_infection": {
        "keywords": ["covid", "viral", "ground-glass", "ground glass", "interstitial", "bilateral infiltrate"],
        "perturbations": {
            "WBC": (0.6, 0.9), "LYMPH_PCT": (0.3, 0.6), "NEUT_PCT": (1.0, 1.3),
            "CRP": (5.0, 30.0), "ESR": (2.0, 4.0),
        },
    },
    "heart_failure": {
        "keywords": ["cardiomegaly", "heart failure", "pulmonary edema", "congestion", "cephalization", "kerley"],
        "perturbations": {
            "BUN": (1.5, 2.5), "CREATININE": (1.3, 2.0), "SODIUM": (0.92, 0.97),
            "BICARB": (0.85, 0.95),
        },
    },
    "effusion": {
        "keywords": ["effusion", "costophrenic", "blunting", "meniscus"],
        "perturbations": {
            "CRP": (3.0, 15.0), "ESR": (2.0, 4.0), "HGB": (0.80, 0.95),
        },
    },
    "anemia_hypoxia": {
        "keywords": ["anemia", "hypoxia", "atelectasis"],
        "perturbations": {
            "HGB": (0.55, 0.75), "HCT": (0.60, 0.80), "RBC": (0.65, 0.80),
            "MCV": (0.80, 0.90),
        },
    },
}


def synthesize_labs(caption: str) -> Tuple[List[float], List[float]]:
    """Generate 20 lab values + 4 one-hot status flags correlated to
    the clinical content in the ROCO caption.

    Returns:
        values_normalized: List[float] of length 20 (z-score normalized)
        flags_onehot:      List[float] of length 4 [is_low, is_normal, is_high, is_critical]
    """
    lower = caption.lower()

    # Start with normal values
    raw_values = {}
    for k, (mean, std) in LAB_REFERENCE.items():
        raw_values[k] = random.gauss(mean, std * 0.1)  # tight around normal

    # Apply clinical perturbations based on caption keywords
    matched_profiles = []
    for profile_name, profile in CLINICAL_PROFILES.items():
        if any(kw in lower for kw in profile["keywords"]):
            matched_profiles.append(profile_name)
            for lab_key, (lo_mult, hi_mult) in profile["perturbations"].items():
                mean, std = LAB_REFERENCE[lab_key]
                if lo_mult > 2.0:  # Absolute replacement (e.g., CRP jumps from 1 to 20-80)
                    raw_values[lab_key] = random.uniform(lo_mult, hi_mult)
                else:  # Multiplicative perturbation
                    multiplier = random.uniform(lo_mult, hi_mult)
                    raw_values[lab_key] = mean * multiplier

    # Normalize values as z-scores
    values_normalized = []
    abnormal_count = 0
    critical_count = 0
    for k in LAB_KEYS:
        mean, std = LAB_REFERENCE[k]
        z = (raw_values[k] - mean) / max(std, 1e-6)
        values_normalized.append(z)
        if abs(z) > 3.0:
            critical_count += 1
        elif abs(z) > 1.5:
            abnormal_count += 1

    # Aggregate flags: [is_low, is_normal, is_high, is_critical]
    low_count = sum(1 for z in values_normalized if z < -1.5)
    high_count = abnormal_count - low_count + (sum(1 for z in values_normalized if 1.5 < z <= 3.0))
    total = len(LAB_KEYS)
    normal_ratio = max(0, total - abnormal_count - critical_count) / total
    flags_onehot = [
        low_count / total,         # fraction of low values
        normal_ratio,              # fraction of normal values
        high_count / total,        # fraction of high values
        critical_count / total,    # fraction of critical values
    ]

    return values_normalized, flags_onehot


def format_lab_string(values_norm: List[float]) -> str:
    """Build a human-readable lab summary string for the prompt."""
    parts = []
    for i, k in enumerate(LAB_KEYS):
        mean, std = LAB_REFERENCE[k]
        actual = values_norm[i] * std + mean  # de-normalize for display
        parts.append(f"{k}={actual:.1f}")
    return ", ".join(parts)


class ROCOMultimodalDataset(Dataset):
    """ROCO-radiology dataset with correlated 20-biomarker lab synthesis
    and Mistral-format instruction prompts with proper -100 label masking."""

    def __init__(self, hf_dataset, transform, tokenizer, max_seq_len=512, limit=-1):
        if limit > 0 and limit < len(hf_dataset):
            self.dataset = hf_dataset.select(range(limit))
        else:
            self.dataset = hf_dataset
        self.transform = transform
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        # ── Image ────────────────────────────────────────────────────
        try:
            img = item["image"].convert("RGB")
            img_tensor = self.transform(img)
        except Exception:
            # Fallback: black image if loading fails
            img_tensor = torch.zeros(3, 224, 224)

        # ── Caption / Findings ───────────────────────────────────────
        caption = item.get("caption", "") or ""
        if not caption.strip():
            caption = "Chest radiograph — no acute abnormality detected."

        # ── Correlated Lab Synthesis ─────────────────────────────────
        lab_values, lab_flags = synthesize_labs(caption)
        lab_input = torch.tensor(lab_values + lab_flags, dtype=COMPUTE_DTYPE)  # (24,)
        lab_string = format_lab_string(lab_values)

        # ── Separate Instruction and Response ────────────────────────
        instruction = (
            f"<s>[INST] <<SYS>>\n"
            f"You are an expert Radiologist and Clinical AI Specialist. "
            f"Correlate the chest radiograph findings with the laboratory panel "
            f"to produce a structured radiology report following RadLex anatomical "
            f"nomenclature and ACR Appropriateness Criteria.\n"
            f"<</SYS>>\n\n"
            f"[LABS: {lab_string}]\n"
            f"Examine the chest scan and laboratory findings to generate "
            f"the Clinical Findings and Impression. [/INST]\n"
        )
        response = (
            f"FINDINGS: {caption}\n"
            f"IMPRESSION: Radiological examination correlated with laboratory panel. </s>"
        )

        # ── Tokenize separately so prompt tokens are 100% masked with -100 ───
        inst_ids = self.tokenizer.encode(instruction, add_special_tokens=False)
        resp_ids = self.tokenizer.encode(response, add_special_tokens=False)

        # Ensure total sequence length fits within max_seq_len
        available_resp = max(32, self.max_seq_len - len(inst_ids))
        if len(resp_ids) > available_resp:
            resp_ids = resp_ids[:available_resp]

        full_ids = inst_ids + resp_ids
        if len(full_ids) > self.max_seq_len:
            full_ids = full_ids[:self.max_seq_len]

        # Mask all instruction tokens with -100; ONLY supervise the response
        inst_len = min(len(inst_ids), len(full_ids))
        labels_list = [-100] * inst_len + full_ids[inst_len:]

        # Pad to self.max_seq_len
        pad_len = max(0, self.max_seq_len - len(full_ids))
        pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 0

        input_ids = torch.tensor(full_ids + [pad_id] * pad_len, dtype=torch.long)
        attention_mask = torch.tensor([1] * len(full_ids) + [0] * pad_len, dtype=torch.long)
        labels = torch.tensor(labels_list + [-100] * pad_len, dtype=torch.long)

        return {
            "image": img_tensor,
            "lab_input": lab_input,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


# ══════════════════════════════════════════════════════════════════════
# Load ROCO-radiology and build DataLoader
# ══════════════════════════════════════════════════════════════════════
print("[*] Loading ROCO-radiology dataset from Hugging Face...")
roco_raw = load_dataset(DATASET_ID, split="train")
print(f"    Raw dataset size: {len(roco_raw)} samples")

dataset = ROCOMultimodalDataset(
    roco_raw, preprocess_vis, tokenizer,
    max_seq_len=MAX_SEQ_LEN, limit=MAX_SAMPLES,
)
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    drop_last=True, num_workers=2, pin_memory=True,
)
print(f"[✓] DataLoader ready: {len(dataset)} samples, {len(dataloader)} batches per epoch")
print(f"    Effective batch size: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")

[*] Loading ROCO-radiology dataset from Hugging Face...
    Raw dataset size: 65423 samples
[✓] DataLoader ready: 20000 samples, 20000 batches per epoch
    Effective batch size: 1 × 16 = 16


In [7]:
import bitsandbytes as bnb

print("=" * 70)
print(">>> STAGE 1: Projector Alignment Warmup")
print("    Training ONLY VisionProjector + TabularLabProjector")
print("    LLM and LoRA adapters are FROZEN")
print("=" * 70)

# ── Freeze everything in the LLM ─────────────────────────────────────
for p in peft_llm.parameters():
    p.requires_grad = False

# ── Ensure projectors are trainable ──────────────────────────────────
for p in vis_proj.parameters():
    p.requires_grad = True
for p in lab_proj.parameters():
    p.requires_grad = True

peft_llm.gradient_checkpointing_enable()

optimizer_s1 = bnb.optim.PagedAdamW8bit(
    list(vis_proj.parameters()) + list(lab_proj.parameters()),
    lr=STAGE1_LR,
)

vis_proj.train()
lab_proj.train()
peft_llm.eval()

stage1_losses = []

for epoch in range(STAGE1_EPOCHS):
    optimizer_s1.zero_grad()
    epoch_loss = 0.0
    num_batches = 0

    pbar = tqdm(dataloader, desc=f"Stage 1 Epoch {epoch+1}/{STAGE1_EPOCHS}", leave=True)
    for step, batch in enumerate(pbar):
        imgs       = batch["image"].to(DEVICE)
        lab_input  = batch["lab_input"].to(DEVICE)
        input_ids  = batch["input_ids"].to(DEVICE)
        attn_mask  = batch["attention_mask"].to(DEVICE)
        labels     = batch["labels"].to(DEVICE)

        # ── Vision features (frozen) ─────────────────────────────────
        with torch.no_grad():
            v_feats = vision_tower(imgs)  # (B, 512)
            if v_feats.ndim == 3:
                v_feats = v_feats[:, 0, :]  # CLS token if patches returned

        # ── Project into LLM space ───────────────────────────────────
        with autocast(dtype=COMPUTE_DTYPE):
            v_tokens = vis_proj(v_feats.to(COMPUTE_DTYPE))   # (B, 1, 4096)
            l_tokens = lab_proj(lab_input)                    # (B, 1, 4096)

            # Text embeddings from LLM's embedding layer (no grad through LLM)
            with torch.no_grad():
                t_tokens = peft_llm.get_input_embeddings()(input_ids)  # (B, seq, 4096)

            # Concatenate: [vision_token, lab_token, text_tokens]
            prefix_len = v_tokens.shape[1] + l_tokens.shape[1]  # 2
            multimodal_embeds = torch.cat([v_tokens, l_tokens, t_tokens], dim=1)

            # Extend attention mask and labels for the 2 prefix tokens
            B = imgs.shape[0]
            prefix_mask   = torch.ones(B, prefix_len, device=DEVICE, dtype=attn_mask.dtype)
            prefix_labels = torch.full((B, prefix_len), -100, device=DEVICE, dtype=labels.dtype)
            ext_attn_mask = torch.cat([prefix_mask, attn_mask], dim=1)
            ext_labels    = torch.cat([prefix_labels, labels], dim=1)

            outputs = peft_llm(
                inputs_embeds=multimodal_embeds,
                attention_mask=ext_attn_mask,
                labels=ext_labels,
            )
            loss = outputs.loss / GRAD_ACCUM

        loss.backward()
        epoch_loss += loss.item() * GRAD_ACCUM
        num_batches += 1

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                list(vis_proj.parameters()) + list(lab_proj.parameters()),
                GRAD_CLIP_NORM,
            )
            optimizer_s1.step()
            optimizer_s1.zero_grad()

        # Progress bar update
        vram = torch.cuda.memory_allocated() / (1024**3)
        pbar.set_postfix(loss=f"{loss.item() * GRAD_ACCUM:.4f}", vram=f"{vram:.1f}GB")

    avg_loss = epoch_loss / max(num_batches, 1)
    stage1_losses.append(avg_loss)
    print(f"\n[✓ Stage 1 Epoch {epoch+1}] Average Loss: {avg_loss:.4f}")

# Save intermediate projector checkpoint
torch.save(
    {"vision_projector": vis_proj.state_dict(), "lab_projector": lab_proj.state_dict()},
    CHECKPOINT_DIR / "projectors_stage1.pt",
)
print(f"\n[✓] Stage 1 complete. Projector checkpoint saved.")
gc.collect()
torch.cuda.empty_cache()

>>> STAGE 1: Projector Alignment Warmup
    Training ONLY VisionProjector + TabularLabProjector
    LLM and LoRA adapters are FROZEN


Stage 1 Epoch 1/1: 100%|██████████| 20000/20000 [2:32:04<00:00,  2.19it/s, loss=1.3626, vram=4.9GB]  



[✓ Stage 1 Epoch 1] Average Loss: 1.1791

[✓] Stage 1 complete. Projector checkpoint saved.


In [8]:
from transformers import get_cosine_schedule_with_warmup

print("=" * 70)
print(">>> STAGE 2: Joint Multimodal QLoRA Fine-Tuning")
print("    Training VisionProjector + TabularLabProjector + BioMistral-LoRA")
print(f"    {STAGE2_EPOCHS} epochs, lr={STAGE2_LR} → {STAGE2_MIN_LR} (Cosine Annealing)")
print("=" * 70)

# ── Unfreeze LoRA parameters ─────────────────────────────────────────
for name, param in peft_llm.named_parameters():
    if "lora" in name.lower():
        param.requires_grad = True

# Enable gradient checkpointing to reduce VRAM
peft_llm.gradient_checkpointing_enable()

# ── Gather all trainable parameters ──────────────────────────────────
trainable_params = (
    [p for p in vis_proj.parameters() if p.requires_grad] +
    [p for p in lab_proj.parameters() if p.requires_grad] +
    [p for p in peft_llm.parameters() if p.requires_grad]
)
print(f"    Total trainable parameters: {sum(p.numel() for p in trainable_params):,}")

optimizer_s2 = bnb.optim.PagedAdamW8bit(
    trainable_params, lr=STAGE2_LR, weight_decay=WEIGHT_DECAY,
)

total_optim_steps = (len(dataloader) // GRAD_ACCUM) * STAGE2_EPOCHS
warmup_steps = max(10, int(total_optim_steps * 0.05))
scheduler = get_cosine_schedule_with_warmup(
    optimizer_s2,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optim_steps,
)
print(f"    Optimizer steps : {total_optim_steps} (warmup: {warmup_steps})")

# ── Training Loop ────────────────────────────────────────────────────
best_loss = float("inf")
stage2_losses = []

for epoch in range(STAGE2_EPOCHS):
    vis_proj.train()
    lab_proj.train()
    peft_llm.train()
    optimizer_s2.zero_grad()
    epoch_loss = 0.0
    num_batches = 0

    pbar = tqdm(dataloader, desc=f"Stage 2 Epoch {epoch+1}/{STAGE2_EPOCHS}", leave=True)
    for step, batch in enumerate(pbar):
        imgs       = batch["image"].to(DEVICE)
        lab_input  = batch["lab_input"].to(DEVICE)
        input_ids  = batch["input_ids"].to(DEVICE)
        attn_mask  = batch["attention_mask"].to(DEVICE)
        labels     = batch["labels"].to(DEVICE)

        # ── Vision features (frozen backbone) ────────────────────────
        with torch.no_grad():
            v_feats = vision_tower(imgs)
            if v_feats.ndim == 3:
                v_feats = v_feats[:, 0, :]

        # ── Forward pass with autocast ───────────────────────────────
        with autocast(dtype=COMPUTE_DTYPE):
            v_tokens = vis_proj(v_feats.to(COMPUTE_DTYPE))  # (B, 1, 4096)
            l_tokens = lab_proj(lab_input)                   # (B, 1, 4096)
            t_tokens = peft_llm.get_input_embeddings()(input_ids)  # (B, seq, 4096)

            prefix_len = v_tokens.shape[1] + l_tokens.shape[1]
            multimodal_embeds = torch.cat([v_tokens, l_tokens, t_tokens], dim=1)

            B = imgs.shape[0]
            prefix_mask   = torch.ones(B, prefix_len, device=DEVICE, dtype=attn_mask.dtype)
            prefix_labels = torch.full((B, prefix_len), -100, device=DEVICE, dtype=labels.dtype)
            ext_attn_mask = torch.cat([prefix_mask, attn_mask], dim=1)
            ext_labels    = torch.cat([prefix_labels, labels], dim=1)

            outputs = peft_llm(
                inputs_embeds=multimodal_embeds,
                attention_mask=ext_attn_mask,
                labels=ext_labels,
            )
            loss = outputs.loss / GRAD_ACCUM

        loss.backward()
        epoch_loss += loss.item() * GRAD_ACCUM
        num_batches += 1

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, GRAD_CLIP_NORM)
            optimizer_s2.step()
            scheduler.step()
            optimizer_s2.zero_grad()

        vram = torch.cuda.memory_allocated() / (1024**3)
        current_lr = scheduler.get_last_lr()[0] if (step + 1) >= GRAD_ACCUM else STAGE2_LR
        pbar.set_postfix(
            loss=f"{loss.item() * GRAD_ACCUM:.4f}",
            lr=f"{current_lr:.2e}",
            vram=f"{vram:.1f}GB",
        )

    avg_loss = epoch_loss / max(num_batches, 1)
    stage2_losses.append(avg_loss)
    print(f"\n[✓ Epoch {epoch+1}/{STAGE2_EPOCHS}] Average Loss: {avg_loss:.4f}")

    # ── Save best checkpoint ─────────────────────────────────────────
    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"    ★ New best loss: {best_loss:.4f} — Saving checkpoint...")
        torch.save(
            {
                "vision_projector": vis_proj.state_dict(),
                "lab_projector": lab_proj.state_dict(),
                "epoch": epoch + 1,
                "best_loss": best_loss,
                "stage1_losses": stage1_losses,
                "stage2_losses": stage2_losses,
            },
            PROJECTORS_PATH,
        )
        peft_llm.save_pretrained(str(LORA_DIR))
        tokenizer.save_pretrained(str(LORA_DIR))
        print(f"    Saved → {PROJECTORS_PATH}")
        print(f"    Saved → {LORA_DIR}/")

    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "=" * 70)
print(f"ALL TRAINING COMPLETED — Best Loss: {best_loss:.4f}")
print("Stage 1 losses:", [f"{l:.4f}" for l in stage1_losses])
print("Stage 2 losses:", [f"{l:.4f}" for l in stage2_losses])
print("=" * 70)

>>> STAGE 2: Joint Multimodal QLoRA Fine-Tuning
    Training VisionProjector + TabularLabProjector + BioMistral-LoRA
    2 epochs, lr=0.00015 → 1e-05 (Cosine Annealing)
    Total trainable parameters: 55,610,368
    Optimizer steps : 2500 (warmup: 125)


Stage 2 Epoch 1/2: 100%|██████████| 20000/20000 [3:55:01<00:00,  1.42it/s, loss=0.9122, lr=8.12e-05, vram=4.9GB]  



[✓ Epoch 1/2] Average Loss: 1.1050
    ★ New best loss: 1.1050 — Saving checkpoint...
    Saved → checkpoints/best_radiology_gpt/projectors.pt
    Saved → checkpoints/best_radiology_gpt/lora_adapter/


Stage 2 Epoch 2/2: 100%|██████████| 20000/20000 [3:55:20<00:00,  1.42it/s, loss=0.1694, lr=0.00e+00, vram=4.9GB]  



[✓ Epoch 2/2] Average Loss: 0.8245
    ★ New best loss: 0.8245 — Saving checkpoint...
    Saved → checkpoints/best_radiology_gpt/projectors.pt
    Saved → checkpoints/best_radiology_gpt/lora_adapter/

ALL TRAINING COMPLETED — Best Loss: 0.8245
Stage 1 losses: ['1.1791']
Stage 2 losses: ['1.1050', '0.8245']


In [9]:
import zipfile, os

print("[*] Packaging trained weights into best_radiology_gpt.zip ...")

with zipfile.ZipFile(str(ZIP_PATH), "w", zipfile.ZIP_DEFLATED) as zf:
    # Add projectors.pt
    zf.write(str(PROJECTORS_PATH), "projectors.pt")
    print(f"    Added: projectors.pt")

    # Add all files in lora_adapter/
    for root, dirs, files in os.walk(str(LORA_DIR)):
        for fname in files:
            full_path = os.path.join(root, fname)
            arcname = os.path.join("lora_adapter", os.path.relpath(full_path, str(LORA_DIR)))
            zf.write(full_path, arcname)
            print(f"    Added: {arcname}")

zip_size_mb = os.path.getsize(str(ZIP_PATH)) / (1024 * 1024)
print(f"\n[✓] Archive created: {ZIP_PATH} ({zip_size_mb:.1f} MB)")

# ── Auto-download in Google Colab ────────────────────────────────────
try:
    from google.colab import files
    files.download(str(ZIP_PATH))
    print("[✓] Browser download triggered (Colab).")
except ImportError:
    print(f"\nNot running in Colab. Download '{ZIP_PATH}' from the file browser.")
except Exception as e:
    print(f"Auto-download failed ({e}). Download '{ZIP_PATH}' manually.")

[*] Packaging trained weights into best_radiology_gpt.zip ...
    Added: projectors.pt
    Added: lora_adapter/chat_template.jinja
    Added: lora_adapter/adapter_config.json
    Added: lora_adapter/tokenizer.json
    Added: lora_adapter/adapter_model.safetensors
    Added: lora_adapter/tokenizer_config.json
    Added: lora_adapter/README.md

[✓] Archive created: best_radiology_gpt.zip (169.4 MB)

Not running in Colab. Download 'best_radiology_gpt.zip' from the file browser.


In [10]:
print("=" * 70)
print(">>> INFERENCE VERIFICATION")
print("    Testing the trained model with a sample from the dataset")
print("=" * 70)

# ── Load a test sample ───────────────────────────────────────────────
test_sample = dataset[0]
test_img       = test_sample["image"].unsqueeze(0).to(DEVICE)
test_lab_input = test_sample["lab_input"].unsqueeze(0).to(DEVICE)

# ── Build inference prompt (instruction only, no answer) ─────────────
test_lab_values = test_sample["lab_input"][:20].tolist()
test_lab_string = format_lab_string(test_lab_values)

inference_prompt = (
    f"<s>[INST] <<SYS>>\n"
    f"You are an expert Radiologist and Clinical AI Specialist. "
    f"Correlate the chest radiograph findings with the laboratory panel "
    f"to produce a structured radiology report following RadLex anatomical "
    f"nomenclature and ACR Appropriateness Criteria.\n"
    f"<</SYS>>\n\n"
    f"[LABS: {test_lab_string}]\n"
    f"Examine the chest scan and laboratory findings to generate "
    f"the Clinical Findings and Impression. [/INST]\n"
)

# ── Tokenize ─────────────────────────────────────────────────────────
tok_out = tokenizer(
    inference_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_SEQ_LEN,
    padding=False,
)
prompt_ids  = tok_out["input_ids"].to(DEVICE)
prompt_mask = tok_out["attention_mask"].to(DEVICE)

# ── Build multimodal input embeddings & Generate ─────────────────────
vis_proj.eval()
lab_proj.eval()
peft_llm.eval()
peft_llm.config.use_cache = True  # Enable KV cache for autoregressive generation

with torch.no_grad(), autocast(dtype=COMPUTE_DTYPE):
    # Vision & Lab prefix tokens
    v_feats = vision_tower(test_img)
    if v_feats.ndim == 3:
        v_feats = v_feats[:, 0, :]
        
    v_tokens = vis_proj(v_feats.to(COMPUTE_DTYPE))
    if v_tokens.ndim == 2:
        v_tokens = v_tokens.unsqueeze(1)  # (1, 1, 4096)

    l_tokens = lab_proj(test_lab_input)
    if l_tokens.ndim == 2:
        l_tokens = l_tokens.unsqueeze(1)  # (1, 1, 4096)

    t_tokens = peft_llm.get_input_embeddings()(prompt_ids)  # (1, seq_len, 4096)

    # Combine into single multimodal embedding sequence
    multimodal_embeds = torch.cat([v_tokens, l_tokens, t_tokens], dim=1)
    prefix_len = v_tokens.shape[1] + l_tokens.shape[1]
    
    ext_mask = torch.cat([
        torch.ones(1, prefix_len, device=DEVICE, dtype=prompt_mask.dtype),
        prompt_mask,
    ], dim=1)

    # Autoregressive generation inside autocast context
    generated = peft_llm.generate(
        inputs_embeds=multimodal_embeds,
        attention_mask=ext_mask,
        max_new_tokens=400,
        do_sample=True,
        temperature=0.2,             # Lower temperature for clinical accuracy
        top_p=0.9,
        repetition_penalty=1.15,     # Avoid repetition loops
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# ── Decode & Clean Output ────────────────────────────────────────────
raw_text = tokenizer.decode(generated[0], skip_special_tokens=True).strip()

# Extract only generated content if prompt tags are present
if "[/INST]" in raw_text:
    output_text = raw_text.split("[/INST]")[-1].strip()
elif "<</SYS>>" in raw_text:
    output_text = raw_text.split("<</SYS>>")[-1].strip()
else:
    output_text = raw_text.strip()

print("\n" + "-" * 70)
print("GENERATED RADIOLOGY REPORT:")
print("-" * 70)
print(output_text)
print("-" * 70)
print("\n[✓] Inference verification complete.")
print("\n" + "=" * 70)
print("TRAINING PIPELINE FINISHED SUCCESSFULLY")
print(f"Download '{ZIP_PATH}' and extract into your local project's weights/ folder:")
print("    weights/")
print("    ├── projectors.pt")
print("    └── lora_adapter/")
print("        ├── adapter_config.json")
print("        ├── adapter_model.safetensors")
print("        ├── tokenizer.json")
print("        └── ...")
print("=" * 70)

>>> INFERENCE VERIFICATION
    Testing the trained model with a sample from the dataset

----------------------------------------------------------------------
GENERATED RADIOLOGY REPORT:
----------------------------------------------------------------------
FINDINGS:  Computed tomography scan of the paranasal sinuses showing a large mass in the left maxillary sinus with destruction of the bony walls

IMPRESSION: Radiological examination correlated with laboratory panel.
----------------------------------------------------------------------

[✓] Inference verification complete.

TRAINING PIPELINE FINISHED SUCCESSFULLY
Download 'best_radiology_gpt.zip' and extract into your local project's weights/ folder:
    weights/
    ├── projectors.pt
    └── lora_adapter/
        ├── adapter_config.json
        ├── adapter_model.safetensors
        ├── tokenizer.json
        └── ...
